# 结构化输出 Structured Output

> 让 LLM 返回格式化的 JSON 数据，而非纯文本。

## 1. with_structured_output：最推荐的方式（基于 Pydantic）

In [1]:
import os
import json
import dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage
from rich import print as rprint

dotenv.load_dotenv()

# 定义输出结构（Pydantic 模型）
class Joke(BaseModel):
    """生成一个笑话。"""
    setup: str = Field(description="笑话的铺垫")
    punchline: str = Field(description="笑话的包袱")
    rating: int = Field(description="笑话评分，1-10")

# 初始化 LLM
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0.7,
)

# with_structured_output：将 LLM 的输出限制为指定的 Pydantic 结构
structured_llm = llm.with_structured_output(Joke)

# 发送请求并手动解析，避免 reasoning_content 干扰
response = llm.invoke("给我讲一个关于程序员的冷笑话，请严格按照 JSON 格式输出 {setup, punchline, rating} 三个字段，不要加任何多余文字。")

# 从 response.content 中提取 JSON
content = response.content
# 去除 markdown 代码块标记
if "```" in content:
    content = content.split("```")[1].strip()
    if content.startswith("json"):
        content = content[4:].strip()

result = Joke.model_validate_json(content)
rprint(f"铺垫：{result.setup}")
rprint(f"包袱：{result.punchline}")
rprint(f"评分：{result.rating}")
rprint(f"\n类型：{type(result)}")

铺垫：为什么程序员总是分不清万圣节和圣诞节？

包袱：因为 Oct 31 == Dec 25。

评分：4

类型：<class '__main__.Joke'>

## 2. with_structured_output + method="json_mode"

In [6]:
import os
import dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

class MovieReview(BaseModel):
    """电影评论分析。"""
    title: str = Field(description="电影名称")
    score: float = Field(description="评分，0-10")
    summary: str = Field(description="简短影评摘要")
    tags: list[str] = Field(description="标签列表，如 ['科幻', '动作']")

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0.7,
)

# method="json_mode"：使用 LLM 的 JSON mode 能力（不依赖 tool calling）
# 适用于不支持 tool calling 的模型
structured_llm = llm.with_structured_output(MovieReview, method="json_mode")

result = structured_llm.invoke("评价电影《星际穿越》")
rprint(f"电影：{result.title}")
rprint(f"评分：{result.score}")
rprint(f"摘要：{result.summary}")
rprint(f"标签：{result.tags}")

OutputParserException: Invalid json output: {"id": "chatcmpl-e38d4bcd-125b-45c4-85e4-9e9acac4aef5", "movie": "Interstellar", "movie_cn": "星际穿越", "director": "Christopher Nolan", "year": 2014, "genre": ["Science Fiction", "Adventure", "Drama"], "theme": ["Love transcends time and space", "Human survival and exploration", "Parent-child relationship", "Science vs. Faith"], "synopsis": "In a near-future Earth plagued by crop blights and dust storms, former NASA pilot Cooper is recruited for a secret mission through a wormhole near Saturn to find a habitable planet for humanity. He must leave behind his young daughter Murph, who grows up feeling abandoned. The mission involves visiting planets near a supermassive black hole named Gargantua, where time dilation effects create heartbreaking separations. Ultimately, Cooper sacrifices himself by falling into the black hole, only to discover that the mysterious "ghost" communicating with Murph was himself, manipulating gravity across time. He su

## 3. with_structured_output：嵌套结构

In [ ]:
import os
import dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

# 嵌套的 Pydantic 模型：结构可以很复杂
class Person(BaseModel):
    """人物信息。"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")

class Relationship(BaseModel):
    """人物关系。"""
    person1: Person = Field(description="第一个人")
    person2: Person = Field(description="第二个人")
    relation: str = Field(description="关系描述，如'父子'、'同事'")
    description: str = Field(description="关系详细说明")

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0.7,
)

structured_llm = llm.with_structured_output(Relationship, method="json_mode")

result = structured_llm.invoke("请描述《三国演义》中诸葛亮和刘备的关系")
rprint(result.model_dump())

## 4. PydanticOutputParser：传统方式

In [ ]:
import os
import dotenv
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

# 定义输出结构
class Recipe(BaseModel):
    name: str = Field(description="菜名")
    ingredients: list[str] = Field(description="食材列表")
    steps: list[str] = Field(description="烹饪步骤")
    cook_time: int = Field(description="烹饪时间，单位分钟")

# 创建 parser，它会自动生成 format_instructions
parser = PydanticOutputParser(pydantic_object=Recipe)

# 在 prompt 中嵌入 format_instructions，告诉 LLM 应该输出什么格式
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个美食专家。\n请按照以下格式输出：\n{format_instructions}"),
    ("human", "推荐一道简单的中餐，食材用常见的。"),
])

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0.7,
)

# 查看自动生成的格式说明
rprint("格式说明：")
rprint(parser.get_format_instructions())

# 使用 LCEL 组合: prompt | llm | parser
chain = prompt | llm | parser

result = chain.invoke({"format_instructions": parser.get_format_instructions()})
rprint(f"\n菜名：{result.name}")
rprint(f"食材：{result.ingredients}")
rprint(f"步骤：{result.steps}")
rprint(f"烹饪时间：{result.cook_time}分钟")

## 5. JsonOutputParser：最简单的 JSON 输出

In [ ]:
import os
import dotenv
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

# JsonOutputParser：不依赖 Pydantic，直接解析 JSON
parser = JsonOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个代码专家。请严格按照 JSON 格式输出，不要添加任何解释、说明或多余文字。\n{format_instructions}"),
    ("human", "分析一下 Python 语言的优缺点，输出 name、 pros、 cons 三个字段"),
])

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0.7,
)

chain = prompt | llm | parser
result = chain.invoke({"format_instructions": parser.get_format_instructions()})

rprint(f"结果类型：{type(result)}")
rprint(f"结果内容：{result}")

## 6. OutputFixingParser：自动修复格式错误的输出

In [ ]:
import os
import dotenv
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser, OutputFixingParser
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

class Weather(BaseModel):
    city: str = Field(description="城市名称")
    temperature: float = Field(description="温度，摄氏度")
    condition: str = Field(description="天气状况，如晴天、多云")
    humidity: int = Field(description="湿度百分比")

# 正常的 parser
parser = PydanticOutputParser(pydantic_object=Weather)

# OutputFixingParser：当 LLM 输出格式不对时，自动用 LLM 修复
# 第一个参数是原始 parser，第二个是用于修复的 LLM
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)

fixing_parser = OutputFixingParser.from_llm(parser=parser, llm=llm)

# 模拟一个格式错误的输出（缺少逗号、字段名拼错）
bad_json = '''{
    "city": "北京"
    "temperature": 25.5
    "condition": "晴天"
    "humidity": 60
}'''

try:
    # 普通 parser 会报错
    result = parser.parse(bad_json)
    rprint(result)
except Exception as e:
    rprint(f"[red]普通 parser 报错：{e}[/red]")
    
    # OutputFixingParser 自动修复
    rprint("\n[green]尝试自动修复...[/green]")
    try:
        fixed_result = fixing_parser.parse(bad_json)
        rprint(f"修复后：{fixed_result}")
    except Exception as e2:
        rprint(f"[red]修复也失败了：{e2}[/red]")

## 7. with_structured_output + 枚举 & 可选字段

In [ ]:
import os
from enum import Enum
import dotenv
from pydantic import BaseModel, Field
from typing import Optional
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

# 使用枚举类型限制字段取值范围
class Sentiment(str, Enum):
    POSITIVE = "正面"
    NEUTRAL = "中性"
    NEGATIVE = "负面"

class Topic(str, Enum):
    TECHNOLOGY = "科技"
    SPORTS = "体育"
    ENTERTAINMENT = "娱乐"
    POLITICS = "政治"
    OTHER = "其他"

class TextAnalysis(BaseModel):
    """文本分析结果。"""
    summary: str = Field(description="文本摘要")
    sentiment: Sentiment = Field(description="情感倾向")
    topic: Topic = Field(description="主题分类")
    keywords: list[str] = Field(description="关键词列表，3-5个")
    score: float = Field(description="综合评分，0-100")
    error: Optional[str] = Field(default=None, description="如果分析失败，描述原因")

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0.7,
)

structured_llm = llm.with_structured_output(TextAnalysis, method="json_mode")

result = structured_llm.invoke("苹果公司今天发布了新一代iPhone，搭载了更强大的A18芯片和全新的相机系统。")
rprint(result.model_dump())

---

# Pydantic 高级特性详解

> 以下六种 Pydantic 特性在 LangChain 结构化输出中极为实用。

## 8. 可选字段 — Optional

> `Optional[X]` 表示字段可以为 `None`，LLM 不需要每次都提供。通常搭配 `default=None`。

In [1]:
import os
import dotenv
from pydantic import BaseModel, Field
from typing import Optional
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

class BookInfo(BaseModel):
    """书籍信息。isbn 和 rating 是可选的，LLM 不知道时可以省略。"""
    title: str = Field(description="书名")
    author: str = Field(description="作者")
    year: int = Field(description="出版年份")
    isbn: Optional[str] = Field(default=None, description="ISBN 编号，可选")
    rating: Optional[float] = Field(default=None, description="评分，0-10，可选")

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)

structured_llm = llm.with_structured_output(BookInfo)

# LLM 可能不知道 ISBN，rating 也可能为空
result = structured_llm.invoke("介绍一下《三体》这本书")
rprint(f"书名：{result.title}")
rprint(f"作者：{result.author}")
rprint(f"年份：{result.year}")
rprint(f"ISBN：{result.isbn}")     # 可能是 None
rprint(f"评分：{result.rating}")     # 可能是 None

# 检查哪些字段为 None
for field_name, value in result:
    if value is None:
        rprint(f"  [yellow]字段 {field_name} 为 None（LLM 未提供）[/yellow]")

ValidationError: 1 validation error for BookInfo
  Invalid JSON: key must be a string at line 1 column 2 [type=json_invalid, input_value='{Here is the user\'s que...类似作品吗？ 😊', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid

## 9. 默认值 — Field(default=...)

> 为字段设置默认值，LLM 未提供时自动填充。适用于语言、单位、地区等不易变的参数。

In [ ]:
import os
import dotenv
from pydantic import BaseModel, Field
from typing import Optional
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

class WeatherReport(BaseModel):
    """天气预报。unit 有默认值 '摄氏度'，不需要 LLM 每次都输出。"""
    city: str = Field(description="城市名称")
    temperature: float = Field(description="温度数值")
    unit: str = Field(default="摄氏度", description="温度单位")
    humidity: Optional[int] = Field(default=0, description="湿度百分比")
    wind: str = Field(default="无风", description="风力描述")

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)

structured_llm = llm.with_structured_output(WeatherReport)

result = structured_llm.invoke("今天北京的天气如何？")
rprint(f"城市：{result.city}")
rprint(f"温度：{result.temperature} {result.unit}")  # unit 使用默认值
rprint(f"湿度：{result.humidity}%")
rprint(f"风力：{result.wind}")

## 10. 枚举类型 — str + Enum

> 限制字段只能取枚举中定义的几个值，LLM 不会输出非法值。比自由字符串更可控。

In [ ]:
import os
from enum import Enum
import dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

# 定义枚举：所有可选值
class Category(str, Enum):
    TECHNOLOGY = "科技"
    LIFESTYLE = "生活"
    EDUCATION = "教育"
    ENTERTAINMENT = "娱乐"

class Priority(str, Enum):
    HIGH = "高"
    MEDIUM = "中"
    LOW = "低"

class TaskInfo(BaseModel):
    """任务分类。category 和 priority 都受枚举约束。"""
    title: str = Field(description="任务标题")
    category: Category = Field(description="任务分类")
    priority: Priority = Field(description="优先级")
    estimate_hours: float = Field(description="预估工时（小时）")

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)

structured_llm = llm.with_structured_output(TaskInfo)

result = structured_llm.invoke("开发一个登录页面，预计需要16小时")
rprint(f"标题：{result.title}")
rprint(f"分类：{result.category.value}（枚举值：{result.category}）")
rprint(f"优先级：{result.priority.value}")
rprint(f"预估工时：{result.estimate_hours}h")

## 11. 列表提取 — list[T]

> LLM 可以一次性提取多个实体到列表中。支持 `list[str]`、`list[int]`、`list[嵌套模型]` 等。

In [ ]:
import os
import dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

# 单个实体模型
class Entity(BaseModel):
    """提取的实体。"""
    name: str = Field(description="实体名称")
    type: str = Field(description="实体类型，如 人名、地名、组织名")

# 外层模型，内部使用 list[T] 提取多个实体
class ExtractionResult(BaseModel):
    """信息抽取结果。"""
    topic: str = Field(description="文本主旨")
    entities: list[Entity] = Field(description="提取的实体列表")
    keywords: list[str] = Field(description="关键词列表")
    numbers: list[int] = Field(description="文中出现的数字列表")

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)

structured_llm = llm.with_structured_output(ExtractionResult)

result = structured_llm.invoke(
    "2024年，OpenAI发布了GPT-4o，苹果推出了iPhone 16，马斯克的SpaceX完成了第300次火箭回收。"
)
rprint(f"主旨：{result.topic}")
rprint(f"\n实体列表（共 {len(result.entities)} 个）：")
for e in result.entities:
    rprint(f"  - {e.name}（{e.type}）")
rprint(f"\n关键词：{result.keywords}")
rprint(f"数字：{result.numbers}")

## 12. 嵌套结构 — 多层 Pydantic 模型

> 模型内部嵌套另一个模型，适应复杂的分层数据结构。

In [ ]:
import os
import dotenv
from pydantic import BaseModel, Field
from typing import Optional
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

# 三层嵌套结构
class Address(BaseModel):
    """地址信息。"""
    street: str = Field(description="街道")
    city: str = Field(description="城市")
    country: str = Field(default="中国", description="国家")

class Department(BaseModel):
    """部门信息。"""
    name: str = Field(description="部门名称")
    head_count: int = Field(description="部门人数")
    address: Address = Field(description="部门地址")

class Company(BaseModel):
    """公司信息。包含嵌套的部门和地址。"""
    name: str = Field(description="公司名称")
    founded: int = Field(description="成立年份")
    headquarters: Address = Field(description="总部地址")
    departments: list[Department] = Field(description="部门列表")

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)

structured_llm = llm.with_structured_output(Company)

result = structured_llm.invoke("阿里巴巴集团成立于1999年，总部在杭州，设有电商、云计算两个核心部门，电商部20000人位于杭州，云计算部15000人位于北京。")
rprint(result.model_dump())

## 13. 限制条件 — Field(ge=, le=, min_length=, max_length=, pattern=)

> Pydantic 内置校验器，自动对 LLM 输出的字段值做合法性检查，不合法则报错。

In [ ]:
import os
import dotenv
from pydantic import BaseModel, Field, field_validator
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

class ValidatedProduct(BaseModel):
    """产品信息，各字段都有校验条件。"""
    name: str = Field(
        description="产品名称",
        min_length=1,        # 至少 1 个字符
        max_length=50,       # 最多 50 个字符
    )
    price: float = Field(
        description="价格（元）",
        ge=0.01,             # ≥ 0.01
        le=999999.99,        # ≤ 999999.99
    )
    quantity: int = Field(
        description="库存数量",
        ge=0,                # ≥ 0
        le=100000,           # ≤ 100000
    )
    rating: float = Field(
        description="评分",
        ge=0.0,              # ≥ 0.0
        le=5.0,              # ≤ 5.0
    )
    email: str = Field(
        description="客服邮箱",
        pattern=r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$",  # 正则校验邮箱格式
    )

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)

structured_llm = llm.with_structured_output(ValidatedProduct)

# 正常情况：LLM 输出符合所有限制
try:
    result = structured_llm.invoke("小米14手机，售价3999元，库存5000件，评分4.8，客服邮箱 support@xiaomi.com")
    rprint(result.model_dump())
except Exception as e:
    rprint(f"[red]校验失败：{e}[/red]")

rprint("\n" + "="*50)

# 手动构造一个超出限制的数据，演示校验拦截
try:
    bad = ValidatedProduct(
        name="测试产品",
        price=-10.0,          # 小于 ge=0.01，会报错
        quantity=999999,       # 大于 le=100000，会报错
        rating=6.0,            # 大于 le=5.0，会报错
        email="not-a-email",   # 不匹配正则，会报错
    )
    rprint(bad)
except Exception as e:
    rprint(f"[red]校验拦截：{e}[/red]")

## 14. 综合实战：商品评价分析（六大特性齐上阵）

In [ ]:
import os
from enum import Enum
import dotenv
from pydantic import BaseModel, Field
from typing import Optional
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

# ── 枚举 ──
class SentimentLabel(str, Enum):
    POSITIVE = "好评"
    NEUTRAL = "中评"
    NEGATIVE = "差评"

class CategoryLabel(str, Enum):
    ELECTRONICS = "电子产品"
    CLOTHING = "服装"
    FOOD = "食品"
    BOOK = "图书"
    OTHER = "其他"

# ── 嵌套 ──
class Aspect(BaseModel):
    """评价维度。"""
    name: str = Field(description="维度名称，如 质量、价格、物流")
    score: float = Field(description="该维度评分", ge=0.0, le=10.0)     # ← 限制条件
    comment: str = Field(description="该维度评语", min_length=1)         # ← 限制条件

# ── 主模型（综合全部特性）──
class ReviewAnalysis(BaseModel):
    """商品评价分析。"""
    product: str = Field(description="商品名称")
    category: CategoryLabel = Field(description="商品分类")                          # ← 枚举
    sentiment: SentimentLabel = Field(description="总体情感")                        # ← 枚举
    overall_score: float = Field(description="总体评分", ge=0.0, le=10.0)            # ← 限制条件
    aspects: list[Aspect] = Field(description="各维度评价列表")                       # ← 列表 + 嵌套
    pros: list[str] = Field(description="优点列表")                                   # ← 列表
    cons: list[str] = Field(description="缺点列表")                                   # ← 列表
    summary: str = Field(description="评价摘要", min_length=5, max_length=200)       # ← 限制条件
    recommend: bool = Field(description="是否推荐购买")
    reply: Optional[str] = Field(default=None, description="商家回复，可选")          # ← 可选 + 默认值

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)

structured_llm = llm.with_structured_output(ReviewAnalysis)

review_text = (
    "刚买的华为MatePad Pro，做工很好，屏幕显示效果一流，续航也很给力。"
    "但价格偏高，配件太少，包装有点简陋。总的来说还是挺满意的，推荐购买。"
)

result = structured_llm.invoke(review_text)

rprint(f"商品：{result.product}  [{result.category.value}]")
rprint(f"总体：{result.sentiment.value}（{result.overall_score}/10）")
rprint(f"推荐：{'✅ 推荐' if result.recommend else '❌ 不推荐'}")

rprint("\n维度评分：")
for a in result.aspects:
    rprint(f"  {a.name}: {'⭐' * int(a.score // 2)}{a.score}分 — {a.comment}")

rprint(f"\n优点：{', '.join(result.pros)}")
rprint(f"缺点：{', '.join(result.cons)}")
rprint(f"摘要：{result.summary}")

if result.reply:
    rprint(f"商家回复：{result.reply}")
else:
    rprint("[yellow]商家未回复[/yellow]")

---

# TypedDict 定义结构化输出

> 除了 Pydantic BaseModel，LangChain 也支持 `TypedDict`。语法更轻量，适合不需要校验的场景。

## 15. TypedDict 基础用法

> 直接定义字典结构的字段类型，`with_structured_output` 自动识别。

In [ ]:
import os
import dotenv
from typing import TypedDict
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

# TypedDict：轻量方式定义输出结构
# 无需 import pydantic，只需标注字段类型
class JokeTD(TypedDict):
    """生成一个笑话。"""
    setup: str       # 铺垫
    punchline: str   # 包袱
    rating: int      # 评分

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0.7,
)

# 和 Pydantic 用法完全一致
structured_llm = llm.with_structured_output(JokeTD)

result = structured_llm.invoke("给我讲一个关于程序员的冷笑话")
rprint(f"类型：{type(result)}")         # <class 'dict'>
rprint(f"铺垫：{result['setup']}")
rprint(f"包袱：{result['punchline']}")
rprint(f"评分：{result['rating']}")

## 16. TypedDict 嵌套

> 字段值可以是另一个 TypedDict，适合分层数据结构。

In [3]:
import os
import dotenv
from typing import TypedDict
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

class AddressTD(TypedDict):
    city: str
    district: str

class PersonTD(TypedDict):
    name: str
    age: int
    address: AddressTD   # 嵌套另一个 TypedDict

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0.7,
)

structured_llm = llm.with_structured_output(PersonTD)

result = structured_llm.invoke("介绍一个住在北京海淀区的25岁程序员小明")
rprint(result)
# rprint(f"姓名：{result['name']}")
# rprint(f"年龄：{result['age']}")
# rprint(f"地址：{result['address']['city']} {result['address']['district']}")

OutputParserException: Invalid json output: {
  "action": "introduce_character",
  "object": {
    "name": "小明",
    "age": 25,
  }
}

你好，关于小明的故事，我可以帮你构思。请问你希望这个角色的故事是轻松日常向的，还是有特定的剧情冲突？
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

## 17. TypedDict + total=False（可选字段）

> `total=False` 使所有字段变为可选，LLM 可以省略部分字段。也可以混用 `NotRequired` 精确控制。

In [ ]:
import os
import dotenv
from typing import TypedDict, NotRequired
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

# 方式一：total=True + NotRequired —— 部分字段可选
class ProductTD(TypedDict):
    name: str           # 必填
    price: float        # 必填
    category: str       # 必填
    stock: NotRequired[int]      # 可选（LLM 可以不提供）
    description: NotRequired[str] # 可选

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)

structured_llm = llm.with_structured_output(ProductTD)

result = structured_llm.invoke("华为Mate60手机，售价5999元，属于电子产品")
rprint(f"名称：{result['name']}")
rprint(f"价格：{result['price']}")
rprint(f"分类：{result['category']}")

# NotRequired 字段可能缺失，访问前用 .get() 安全获取
stock = result.get('stock', '未提供')
desc = result.get('description', '未提供')
rprint(f"库存：{stock}")
rprint(f"描述：{desc}")

## 18. TypedDict + list[T]（列表提取）

> TypedDict 配合列表类型提取多个实体，代码更简洁。

In [ ]:
import os
import dotenv
from typing import TypedDict, NotRequired
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

class EntityTD(TypedDict, total=True):
    name: str
    type: str                          # 实体类型：人物/地点/组织
    importance: NotRequired[float]     # 重要度，可选

class NewsExtractionTD(TypedDict):
    title: str                         # 新闻标题
    summary: str                       # 摘要
    entities: list[EntityTD]           # 实体列表
    tags: list[str]                    # 标签列表

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)

structured_llm = llm.with_structured_output(NewsExtractionTD)

result = structured_llm.invoke(
    "2025年6月，特斯拉CEO埃隆·马斯克在德州超级工厂宣布，"
    "特斯拉AI机器人Optimus已开始在工厂试运行，预计年底前量产1000台。"
)
rprint(f"标题：{result['title']}")
rprint(f"摘要：{result['summary']}")
rprint(f"标签：{result['tags']}")
rprint(f"\n实体（共 {len(result['entities'])} 个）：")
for e in result['entities']:
    imp = f' (重要度: {e["importance"]})' if 'importance' in e else ''
    rprint(f"  - {e['name']} ({e['type']}){imp}")

## 19. TypedDict vs Pydantic：怎么选？

| 对比维度 | TypedDict | Pydantic BaseModel |
|---------|-----------|-------------------|
| 导入依赖 | `typing` 内置模块，零依赖 | 需安装 `pydantic` |
| 返回类型 | `dict` | Pydantic 模型对象 |
| 字段描述 | 不支持（无法传 description） | `Field(description=...)` |
| 校验能力 | 无（仅类型提示） | 支持 `ge/le/pattern` 等 |
| 默认值 | `NotRequired` 实现可选 | `Optional` + `Field(default=...)` |
| 枚举约束 | 不支持 | 支持 `str + Enum` |
| 嵌套 | 支持 TypedDict 嵌套 | 支持 Pydantic 模型嵌套 |
| 适用场景 | 快速原型、简单结构 | 生产级、需要严格校验 |

In [ ]:
import os
import dotenv
from typing import TypedDict
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

# ── 同一个任务，两种写法 ──

# TypedDict 版（轻量）
class MovieTD(TypedDict):
    title: str
    year: int
    director: str
    rating: float

# Pydantic 版（带校验与描述）
class MoviePD(BaseModel):
    """电影信息。"""
    title: str = Field(description="电影名称")
    year: int = Field(description="上映年份", ge=1900, le=2030)
    director: str = Field(description="导演")
    rating: float = Field(description="评分", ge=0.0, le=10.0)

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)

# TypedDict 返回 dict
td_result = llm.with_structured_output(MovieTD).invoke("推荐一部诺兰导演的高分电影")
rprint(f"[bold]TypedDict 结果：[/bold]")
rprint(f"  {td_result}")
rprint(f"  类型：{type(td_result).__name__}")
rprint(f"  访问方式：td_result['title'] = {td_result['title']}")

rprint()

# Pydantic 返回模型对象
pd_result = llm.with_structured_output(MoviePD).invoke("推荐一部诺兰导演的高分电影")
rprint(f"[bold]Pydantic 结果：[/bold]")
rprint(f"  {pd_result}")
rprint(f"  类型：{type(pd_result).__name__}")
rprint(f"  访问方式：pd_result.title = {pd_result.title}")

---

# JSON Schema 直接定义结构

> 直接传入一个裸的 JSON Schema 字典，零依赖、无需任何导入。适合动态定义结构的场景。

## 20. JSON Schema 基础用法

> `with_structured_output` 可以直接接受一个 JSON Schema dict，不需要定义类。返回的是 Python `dict`。

In [ ]:
import os
import dotenv
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

# 直接写 JSON Schema 字典
# schema 中的 title 字段会作为 description 传给 LLM
joke_schema = {
    "title": "Joke",
    "description": "生成一个笑话",
    "type": "object",
    "properties": {
        "setup": {"title": "铺垫", "type": "string", "description": "笑话的铺垫"},
        "punchline": {"title": "包袱", "type": "string", "description": "笑话的包袱"},
        "rating": {"title": "评分", "type": "integer", "description": "笑话评分 1-10"},
    },
    "required": ["setup", "punchline", "rating"],
}

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0.7,
)

# 传入 JSON Schema dict
structured_llm = llm.with_structured_output(joke_schema)

result = structured_llm.invoke("讲一个关于猫的笑话")
rprint(f"类型：{type(result)}")   # dict
rprint(f"结果：{result}")

## 21. JSON Schema 嵌套 & 列表

> 通过 `$defs` 定义可复用的子结构，`items` 定义数组元素类型。适用动态生成 schema 的场景。

In [ ]:
import os
import dotenv
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

# 用 $defs 定义可复用的子 schema
movie_schema = {
    "title": "MovieReview",
    "description": "电影评论分析",
    "type": "object",
    "properties": {
        "title": {"type": "string", "description": "电影名称"},
        "year": {"type": "integer", "description": "上映年份"},
        "score": {"type": "number", "description": "评分 0-10"},
        "director": {"type": "string", "description": "导演"},
        "actors": {
            "type": "array",
            "description": "主要演员",
            "items": {"$ref": "#/$defs/Actor"},
        },
    },
    "required": ["title", "year", "score", "director", "actors"],
    "$defs": {
        "Actor": {
            "type": "object",
            "properties": {
                "name": {"type": "string", "description": "演员姓名"},
                "role": {"type": "string", "description": "饰演角色"},
            },
            "required": ["name", "role"],
        }
    },
}

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)

structured_llm = llm.with_structured_output(movie_schema)

result = structured_llm.invoke("评价电影《盗梦空间》，2010年上映，诺兰导演，莱昂纳多主演柯布")
rprint(f"电影：{result['title']} ({result['year']})")
rprint(f"评分：{result['score']}")
rprint(f"导演：{result['director']}")
rprint(f"\n演员：")
for a in result['actors']:
    rprint(f"  - {a['name']} 饰 {a['role']}")

## 22. JSON Schema + enum（枚举约束）

> JSON Schema 原生支持 `enum` 枚举，即使不导入 Pydantic 也能约束字段取值。

In [ ]:
import os
import dotenv
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

# enum 约束：字段只能取指定值
task_schema = {
    "title": "Task",
    "type": "object",
    "properties": {
        "title": {"type": "string", "description": "任务标题"},
        "priority": {
            "type": "string",
            "description": "优先级",
            "enum": ["高", "中", "低"],     # ← 三选一
        },
        "status": {
            "type": "string",
            "description": "状态",
            "enum": ["待开始", "进行中", "已完成"],
        },
        "estimate_hours": {"type": "number", "description": "预估工时"},
    },
    "required": ["title", "priority", "status"],
}

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)

structured_llm = llm.with_structured_output(task_schema)

result = structured_llm.invoke("开发登录页面，预计16小时")
rprint(f"任务：{result['title']}")
rprint(f"优先级：{result['priority']}")
rprint(f"状态：{result['status']}")
rprint(f"工时：{result.get('estimate_hours', '未提供')}h")

---

# @dataclass 定义结构化输出

> Python 内置 `@dataclass` 装饰器，无需任何第三方库，LangChain 的 `with_structured_output` 原生支持。

## 23. @dataclass 基础用法

> 用 `@dataclass` 标注一个类，字段用类型注解。比 Pydantic 更轻量，比 TypedDict 支持方法。

In [2]:
import os
from dataclasses import dataclass, field
import dotenv
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

# @dataclass：内置模块，零依赖
# 字段用类型注解，默认值用 field 或直接赋值
@dataclass
class JokeDC:
    """生成一个笑话。"""
    setup: str             # 铺垫
    punchline: str         # 包袱
    rating: int = 5        # 评分，默认值 5

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0.7,
)

# 和 Pydantic / TypedDict 用法完全一致
structured_llm = llm.with_structured_output(JokeDC)

result = structured_llm.invoke("讲一个关于程序员的冷笑话")
rprint(f"类型：{type(result)}")         # <class '__main__.JokeDC'>
rprint(f"{result}")

# rprint(f"铺垫：{result.setup}")
# rprint(f"包袱：{result.punchline}")
# rprint(f"评分：{result.rating}")

类型：<class 'dict'>

{'content': 
'来一个经典的：\n\n一个程序员被老婆派去买菜：\n\n"去买10个包子，如果有西瓜，买一个。"\n\n程序员回来了，手里拿着**1
个包子**。\n\n老婆："怎么只买一个？！"\n\n程序员："有西瓜啊。"\n\n---\n\n😄 
逻辑上完全正确——如果有西瓜，买一个（包子）。\n\n这就是程序员的思维方式，条件判断里缺了一个括号的下场。', 'role': 
'assistant'}

## 24. @dataclass + field (metadata 描述字段)

> `field(metadata={"description": ...})` 为 dataclass 字段添加描述信息，LLM 能更准确理解字段含义。

In [ ]:
import os
from dataclasses import dataclass, field
from typing import Optional
import dotenv
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

@dataclass
class RecipeDC:
    """食谱信息。"""
    name: str = field(metadata={"description": "菜名"})
    cook_time: int = field(metadata={"description": "烹饪时间（分钟）"})
    difficulty: str = field(metadata={"description": "难度：简单/中等/困难"})
    ingredients: list[str] = field(metadata={"description": "食材列表"})
    steps: list[str] = field(metadata={"description": "烹饪步骤"})
    tips: Optional[str] = field(default=None, metadata={"description": "小贴士，可选"})

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)

structured_llm = llm.with_structured_output(RecipeDC)

result = structured_llm.invoke("推荐一道简单的番茄炒蛋做法")
rprint(f"菜名：{result.name}")
rprint(f"时间：{result.cook_time}分钟")
rprint(f"难度：{result.difficulty}")
rprint(f"食材：{', '.join(result.ingredients)}")
rprint(f"\n步骤：")
for i, step in enumerate(result.steps, 1):
    rprint(f"  {i}. {step}")
if result.tips:
    rprint(f"\n小贴士：{result.tips}")

## 25. @dataclass 嵌套

> dataclass 之间也可以互相嵌套，形成分层结构。

In [ ]:
import os
from dataclasses import dataclass, field
import dotenv
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

@dataclass
class AddressDC:
    city: str = field(metadata={"description": "城市"})
    district: str = field(metadata={"description": "区"})

@dataclass
class EmployeeDC:
    name: str = field(metadata={"description": "姓名"})
    position: str = field(metadata={"description": "职位"})
    salary: int = field(metadata={"description": "年薪（万元）"})
    address: AddressDC = field(metadata={"description": "居住地址"})

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)

structured_llm = llm.with_structured_output(EmployeeDC)

result = structured_llm.invoke("介绍一位在上海浦东工作的Java架构师张三，年薪50万")
rprint(f"姓名：{result.name}")
rprint(f"职位：{result.position}")
rprint(f"年薪：{result.salary}万")
rprint(f"地址：{result.address.city} {result.address.district}")

## 26. 四种方式对比一览
> 推荐使用 Pydantic

| 对比维度 | JSON Schema | @dataclass | TypedDict | Pydantic BaseModel |
|---------|-------------|------------|-----------|-------------------|
| 依赖 | 零依赖 | 内置 `dataclasses` | 内置 `typing` | 需安装 `pydantic` |
| 返回类型 | `dict` | dataclass 实例 | `dict` | 模型对象 |
| 字段描述 | properties 中写 description | `field(metadata={"description": ...})` | ❌ 不支持 | ✅ `Field(description=...)` |
| 校验 | ✅ enum / type 校验 | ❌ 无 | ❌ 无 | ✅ ge/le/pattern 等 |
| 默认值 | ❌ 不支持 | ✅ `field(default=...)` | ❌ (`NotRequired` 仅可选) | ✅ `Field(default=...)` |
| 枚举 | ✅ enum 关键字 | ✅ Enum 配合 | ❌ 不支持 | ✅ str+Enum |
| 嵌套 | ✅ $defs + $ref | ✅ dataclass 嵌套 | ✅ TypedDict 嵌套 | ✅ 模型嵌套 |
| 列表 | ✅ items + $ref | ✅ `list[T]` | ✅ `list[T]` | ✅ `list[T]` |
| 方法/行为 | ❌ 纯数据 | ✅ 可定义方法 | ❌ 纯数据 | ✅ 可定义方法 |
| 适用场景 | 动态 schema / API 响应 | 轻量对象、需方法 | 快速原型 | 生产级校验 |

In [3]:
import os
import dotenv
from dataclasses import dataclass, field
from typing import TypedDict
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from rich import print as rprint

dotenv.load_dotenv()

# ── 同一个任务，四种方式 ──

# 1. JSON Schema
book_schema = {
    "title": "Book",
    "type": "object",
    "properties": {
        "title": {"type": "string", "description": "书名"},
        "author": {"type": "string", "description": "作者"},
        "year": {"type": "integer", "description": "出版年份"},
        "rating": {"type": "number", "description": "评分"},
    },
    "required": ["title", "author", "year", "rating"],
}

# 2. @dataclass
@dataclass
class BookDC:
    title: str = field(metadata={"description": "书名"})
    author: str = field(metadata={"description": "作者"})
    year: int = field(metadata={"description": "出版年份"})
    rating: float = field(metadata={"description": "评分"})

# 3. TypedDict
class BookTD(TypedDict):
    title: str
    author: str
    year: int
    rating: float

# 4. Pydantic
class BookPD(BaseModel):
    """书籍信息。"""
    title: str = Field(description="书名")
    author: str = Field(description="作者")
    year: int = Field(description="出版年份", ge=1900, le=2030)
    rating: float = Field(description="评分", ge=0.0, le=10.0)

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)

topic = "推荐一本刘慈欣写的科幻小说"

# JSON Schema → dict
r1 = llm.with_structured_output(book_schema).invoke(topic)
rprint(f"[bold]1. JSON Schema[/bold] → dict: {r1}")

# @dataclass → dataclass 实例
r2 = llm.with_structured_output(BookDC).invoke(topic)
rprint(f"[bold]2. @dataclass[/bold] → {type(r2).__name__}: {r2}")

# TypedDict → dict
r3 = llm.with_structured_output(BookTD).invoke(topic)
rprint(f"[bold]3. TypedDict[/bold] → dict: {r3}")

# Pydantic → 模型对象
r4 = llm.with_structured_output(BookPD).invoke(topic)
rprint(f"[bold]4. Pydantic[/bold] → {type(r4).__name__}: {r4}")

OutputParserException: Invalid json output: { thinking: The user is asking for a recommendation of a science fiction novel by Liu Cixin. This is a straightforward request, and I can provide a recommendation based on the popularity and acclaim of his works. }

我非常推荐刘慈欣的《三体》三部曲！这是他最负盛名的作品，也是中国科幻文学的里程碑。

**《三体》三部曲包括：**
- 《三体》（第一部）
- 《黑暗森林》（第二部）
- 《死神永生》（第三部）

**为什么推荐这本书：**

1. **宏大的宇宙观**：小说构建了一个跨越数百年、涉及多个文明的宇宙图景，展现了"黑暗森林法则"等令人震撼的宇宙社会学理论。

2. **硬核的科学设定**：书中涉及大量物理学、天文学知识，比如三体问题、降维打击、光速飞船等，既有科学依据又有大胆想象。

3. **深刻的人文思考**：探讨了人类文明的脆弱性、道德选择的困境，以及在宇宙尺度下人类的意义。

4. **世界级影响力**：该系列获得了雨果奖等多项国际大奖，被翻译成多种语言，Netflix也改编成了电视剧。

如果你时间有限，可以先从第一部《三体》读起，感受一下三体人入侵地球前的那段历史背景和科学悬疑的氛围。读完后如果意犹未尽，再继续后面两部，体验会更加完整！
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 